**Predictions of the trained model and graphs**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler

def filter_by_datetime(ozoneprint, start_date, start_hour, end_date, end_hour,chambnumb):
    # Ensure Date column is datetime
    ozoneprint['Date'] = pd.to_datetime(ozoneprint['Date'])

    # Create full datetime column (combine Date + hour)
    ozoneprint['DateTime'] = ozoneprint['Date']# + pd.to_timedelta(ozoneprint['hour'], unit='h')

    # Build start and end datetime
    start_dt = pd.to_datetime(start_date)# + pd.to_timedelta(start_hour, unit='h')
    end_dt = pd.to_datetime(end_date)# + pd.to_timedelta(end_hour, unit='h')

    mask = (
        (ozoneprint['DateTime'] >= start_dt) &
        (ozoneprint['DateTime'] <= end_dt) &
        (ozoneprint['hour'] >= start_hour) &
        (ozoneprint['hour'] <= end_hour) &
        (ozoneprint['Chamber'] == chambnumb)
    )
    return ozoneprint[mask].copy()

def ozoneplots(ozoneprint):

    start_date=input("Set the start date in the format of 2024-10-01):")
    #start_date="2024-10-01"
    start_hour=int(input("Set the start hour (0-23):"))
    #start_hour=12
    end_date=input("Set the end date in the format of 2024-10-01):")
    #end_date="2024-10-20"
    end_hour=int(input("Set the end hour (0-23):"))
    #end_hour=22
    chambnumb=int(input("Set the chamber (1-4):"))
    filtered_df = filter_by_datetime(ozoneprint, start_date, start_hour, end_date, end_hour,chambnumb)

    if filtered_df.empty:
        print("No data in the specified date/hour range.")
        return filtered_df,None

    # --- Step 2: Load ensemble model ---
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/OZONE BEST TRAINED MODELS/New dataset/Mix features"
    #modelname="weightedensemble_model.pkl"
    modelname="GB_test_5_v2_42.pkl"
    featurescaler_path = os.path.join(save_dir, "feature_scaler.pkl")
    targetscaler_path = os.path.join(save_dir, "target_scaler.pkl")
    model_path = os.path.join(save_dir, modelname)
    #ensemble_model = joblib.load(model_path)
    gb_model = joblib.load(model_path)
    feature_scaler = joblib.load(featurescaler_path)
    target_scaler = joblib.load(targetscaler_path)
    #method = ensemble_model.get("method")
    #print("The ensemble model method is: ", method)
    model_features = ["Chamber", "WaterTemperature", "RFTurbidity", "FlowRate", "Ozonedosage"]

    def safe_predict(m, f, df):
        X_sub = df.reindex(columns=f, fill_value=0)
        scaler_features = feature_scaler.feature_names_in_ if hasattr(feature_scaler, "feature_names_in_") else f
        X_full = pd.DataFrame(0, index=df.index, columns=scaler_features)
        common_features = [c for c in scaler_features if c in X_sub.columns]
        X_full[common_features] = X_sub[common_features]
        X_scaled = feature_scaler.transform(X_full)
        model_feature_indices = [list(scaler_features).index(c) for c in f if c in scaler_features]
        X_model = X_scaled[:, model_feature_indices]
        #X_subscaled = feature_scaler.transform(X_sub)
        y_predscaled = m.predict(X_model)
        y_pred = target_scaler.inverse_transform(y_predscaled.reshape(-1, 1))
        y_pred = np.array(y_pred).reshape(-1)  # flatten in case shape is (n,1)
        return y_pred

    def predict_simple(m, f, df):
        # 1. Align features with the scaler first (scaler expects 8 features usually)
        scaler_features = list(feature_scaler.feature_names_in_) if hasattr(feature_scaler, "feature_names_in_") else f
        X_full = pd.DataFrame(0, index=df.index, columns=scaler_features)

        # 2. Fill the columns the model actually uses from filtered_df
        for col in f:
            if col in df.columns:
                X_full[col] = df[col]

        # 3. Scale the full feature set
        X_scaled = feature_scaler.transform(X_full)

        # 4. Filter only the features the model was trained on
        # This matches the indices of the model_features within the scaler_features
        model_feature_indices = [scaler_features.index(c) for c in f]
        X_model = X_scaled[:, model_feature_indices]

        # 5. Predict and Inverse Transform
        y_pred_scaled = m.predict(X_model)
        y_pred_unscaled = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        return y_pred_unscaled


    # --- Step 3: Real predictions ---
    print(f"Predicting using simple model: {modelname}")
    y_pred = predict_simple(gb_model, model_features, filtered_df)

    #if method in ["stacking"]:
    #    meta_model = ensemble_model["meta_model"]
    #    test_preds = np.column_stack([safe_predict(m, f, filtered_df) for m, f in base_models])
    #    y_pred = meta_model.predict(test_preds)

    #elif method in ["bagging"]:
    #    n_bags = ensemble_model.get("n_bags", 1)
    #    bag_preds = np.zeros((len(filtered_df), len(base_models) * n_bags))
    #    idx = 0
    #    for m, features in base_models:
    #        for b in range(n_bags):
    #            bag_preds[:, idx] = safe_predict(m, features, filtered_df)
    #            idx += 1
    #    y_pred = np.mean(bag_preds, axis=1)

    #elif method in ["weighted", "optimised", "multi_optimised"]:
    #    weights = np.array(ensemble_model.get("weights"))
    #    test_preds = np.column_stack([safe_predict(m, f, filtered_df) for m, f in base_models])
    #    y_pred = np.dot(test_preds, weights)

    #else:
    #   raise ValueError("Unknown ensemble method")

    # --- Step 4: Sensitivity Analysis ---
    print("\n--- Sensitivity Analysis ---")
    print(f"Available features: {model_features}")
    feat_to_change = input("Which feature would you like to vary? ")
    changes_str = input("Enter percentage changes separated by commas (e.g. -10, 10, 20): ")
    percent_changes = [float(x.strip()) for x in changes_str.split(',')]

    # Create a copy and modify the specific column
    #perturbed_df = filtered_df.copy()
    #if feat_to_change in perturbed_df.columns:
    #    multiplier = 1 + (percent_change / 100)
    #    perturbed_df[feat_to_change] = perturbed_df[feat_to_change] * multiplier

        # Rerun model on perturbed data
    #    y_pred_perturbed = predict_simple(gb_model, model_features, perturbed_df)
    #else:
    #    print(f"Feature {feat_to_change} not found. Skipping sensitivity plot.")
    #    y_pred_perturbed = None

    # --- Step 5: Plot ---
    plt.figure(figsize=(16,8))
    plt.plot(filtered_df['DateTime'], filtered_df['CT Value EPA calculated'], label="EPA CT", marker='o')
    plt.plot(filtered_df['DateTime'], y_pred, label="Predicted CT", marker='x')
    plt.scatter(filtered_df['DateTime'], filtered_df['CT_All'],color='red', s=100, zorder=5, label="Measured CT")
    colors = plt.cm.coolwarm(np.linspace(0, 1, len(percent_changes)))
    for i, pct in enumerate(percent_changes):
        perturbed_df = filtered_df.copy()
        multiplier = 1 + (pct / 100)
        perturbed_df[feat_to_change] = perturbed_df[feat_to_change] * multiplier

        y_pred_perturbed = predict_simple(gb_model, model_features, perturbed_df)

        plt.plot(filtered_df['DateTime'], y_pred_perturbed,
                 label=f"{feat_to_change} {pct:+}%",
                 color=colors[i], alpha=0.8, linestyle='-')

    #if y_pred_perturbed is not None:
    #    plt.plot(filtered_df['DateTime'], y_pred_perturbed,
    #             label=f"Predicted CT ({feat_to_change} {percent_change:+}%)",
    #             marker='x', color='green', linestyle='-')
    plt.xlabel("DateTime")
    plt.ylabel("CT Value (mg O₃/L * min)")
    plt.title(f"Sensitivity Analysis: Impact of varying {feat_to_change} on CT Prediction")
    #plt.title(f"Ozone CT Predictions vs EPA Calculated Values in chamber{chambnumb}\n{start_date} {start_hour}:00 to {end_date} {end_hour}:00")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') # Move legend outside
    plt.grid(True, alpha=0.3)
    #plt.legend()
    #plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return filtered_df, y_pred